In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
excel_path = "/home/ubuntu/giodir/digitalPathology/data/aiFlopp/REGGIO AIFLOPP Prostata - tabella dati - 050526.xlsx"
sheet_name = "Confronto Reggio-Trento"

In [3]:
# open excel sheet in pandas

df = pd.read_excel(excel_path, sheet_name=sheet_name, header=[0, 1])

/home/ubuntu/giodir/digitalPathology/.venv/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [4]:
df.head()

REFERTI REGGIO                             \
      ID PATIENT CODICE CASO REPERE/VETRINO   
0           RE01  I-25-16291              A   
1           RE01  I-25-16291              B   
2           RE01  I-25-16291              C   
3           RE01  I-25-16291              D   
4           RE01  I-25-16291              E   

                                                  \
  NUMERO FRUSTOLI PER REPERE/VETRINO     LETTORE   
0                                  1  CAVAZZA A.   
1                                  1  CAVAZZA A.   
2                                  1  CAVAZZA A.   
3                                  1  CAVAZZA A.   
4                                  1  CAVAZZA A.   

                                                                                                                                                                                                                                                \
  DIAGNOSI (0=no tumore; 1=ASAP/ATYP che richiede approfondimento; 2=PIN di alto grado; 3=AK intraduttale; 4=positivo per adenocarcinoma acinare; 5=positvo per adenocarcinoma duttale; N.A.=frustolo senza evidenza di ghiandole prostatiche)   
0                                                  4                                                                                                                                                                                             
1                                                  4                                                                                                                                                                                             
2                                                  0                                                                                                                                                                                             
3                                                  4                                                                                                                                                                                             
4                                                  0                                                                                                                                                                                             

                                                                                                                                                                                                                                                            \
  DIAGNOSI DOPO IMMUNO (0=no tumore; 1=ASAP/ATYP che richiede approfondimento; 2=PIN di alto grado; 3=AK intraduttale; 4=positivo per adenocarcinoma acinare; 5=positvo per adenocarcinoma duttale; N.A.=frustolo senza evidenza di ghiandole prostatiche)   
0                                                NaN                                                                                                                                                                                                         
1                                                NaN                                                                                                                                                                                                         
2                                                NaN                                                                                                                                                                                                         
3                                                NaN                                                                                                                                                                                                         
4                                                NaN                                    

In [5]:
df.columns.to_list()

[('REFERTI REGGIO', 'ID PATIENT'),
 ('REFERTI REGGIO', 'CODICE CASO'),
 ('REFERTI REGGIO', 'REPERE/VETRINO'),
 ('REFERTI REGGIO', 'NUMERO FRUSTOLI PER REPERE/VETRINO'),
 ('REFERTI REGGIO', 'LETTORE'),
 ('REFERTI REGGIO',
  'DIAGNOSI (0=no tumore; 1=ASAP/ATYP che richiede approfondimento; 2=PIN di alto grado; 3=AK intraduttale; 4=positivo per adenocarcinoma acinare; 5=positvo per adenocarcinoma duttale; N.A.=frustolo senza evidenza di ghiandole prostatiche)'),
 ('REFERTI REGGIO',
  'DIAGNOSI DOPO IMMUNO (0=no tumore; 1=ASAP/ATYP che richiede approfondimento; 2=PIN di alto grado; 3=AK intraduttale; 4=positivo per adenocarcinoma acinare; 5=positvo per adenocarcinoma duttale; N.A.=frustolo senza evidenza di ghiandole prostatiche)'),
 ('REFERTI REGGIO', 'GLEASON Principale'),
 ('REFERTI REGGIO', 'GLEASON Secondario'),
 ('REFERTI REGGIO', '% PATTERN 3'),
 ('REFERTI REGGIO', '% PATTERN 4'),
 ('REFERTI REGGIO', '% PATTERN 5'),
 ('REFERTI REGGIO', 'LUNGHEZZA CORES BIOPTICI (mm)'),
 ('REFERTI RE

In [6]:
columns_to_keep = [
    ('REFERTI REGGIO', 'ID PATIENT'),
    ('REFERTI REGGIO', 'CODICE CASO'),
    ('REFERTI REGGIO', 'REPERE/VETRINO'),
    ('REFERTI REGGIO', 'GG ISUP PER SINGOLO REPERE/VETRINO'),
    ('REFERTI TRENTO', 'GG ISUP PER SINGOLO REPERE/VETRINO'),
]

columns_new_names = {
    ('REFERTI REGGIO', 'ID PATIENT'): 'patient_id',
    ('REFERTI REGGIO', 'CODICE CASO'): 'case_code',
    ('REFERTI REGGIO', 'REPERE/VETRINO'): 'bersaglio',
    ('REFERTI REGGIO', 'GG ISUP PER SINGOLO REPERE/VETRINO'): 'GG_reggio',
    ('REFERTI TRENTO', 'GG ISUP PER SINGOLO REPERE/VETRINO'): 'GG_trento',
}


normalized_cols = [tuple(x.strip() if isinstance(x, str) else x for x in col)
                   if isinstance(col, tuple) else col
                   for col in df.columns]
df.columns = pd.MultiIndex.from_tuples(normalized_cols)  # only if they are tuples

filtered_df = df[columns_to_keep]
filtered_df.columns = [columns_new_names.get(col, col) for col in filtered_df.columns]

In [7]:
filtered_df.head()

,patient_id,case_code,bersaglio,GG_reggio,GG_trento
0,RE01,I-25-16291,A,1.0,NaN
1,RE01,I-25-16291,B,1.0,NaN
2,RE01,I-25-16291,C,NaN,NaN
3,RE01,I-25-16291,D,3.0,3.0
4,RE01,I-25-16291,E,NaN,NaN


In [8]:
# Assign -1 values to case where GG is missing (meaning there is no tumor)
filtered_df['GG_reggio'] = filtered_df['GG_reggio'].fillna(-1)
filtered_df['GG_trento'] = filtered_df['GG_trento'].fillna(-1)

In [13]:
filtered_df["difference"] = np.abs(filtered_df['GG_reggio'] - filtered_df['GG_trento'])

In [14]:
filtered_df["difference"].value_counts()

difference
0.0    460
1.0     54
2.0     29
3.0      6
4.0      5
5.0      2
Name: count, dtype: int64

In [15]:
len(filtered_df)

556

In [16]:
# Parse labels to get the bag_id

parsed_case_code = filtered_df["case_code"].str.replace('-', '_')
filtered_df['bag_id'] = "RE_" + parsed_case_code + "_1_" + filtered_df["bersaglio"].astype(str)

In [17]:
# Filter to keep only the analyzed cases

features_dir = Path("/home/ubuntu/giodir/digitalPathology/data/uni_features_RE_all")

available_bags = {path.stem for path in features_dir.glob("*.npz")}
print("Analyzed bags:", len(available_bags))

avail_df = filtered_df[filtered_df['bag_id'].isin(available_bags)]

print(f"Total cases: {len(filtered_df)}, Available cases: {len(avail_df)},")

Analyzed bags: 550
Total cases: 556, Available cases: 550,


## LABEL TYPE

In [18]:
## binary like 0 vs 1+
avail_df["binary_difference"] = (avail_df["difference"] > 0).astype(int)

# keep only the important differences (set 0 where the difference is 0, 1 if it is 2 or more and None if it is 1)
avail_df["important_difference"] = avail_df["difference"].apply(lambda x: 0 if x == 0 else (1 if x >= 2 else None))

In [20]:
# Save in csv the three files like (bag_id, label_col)

basedir = Path("/home/ubuntu/giodir/digitalPathology/data/all_discordance_labels")
basedir.mkdir(exist_ok=True)


# binary diff
avail_df[["bag_id", "binary_difference"]].rename(columns={"binary_difference": "label"}).to_csv(
    basedir / "binary_diff_labels.csv", index=False)

# binary important diff
avail_df[["bag_id", "important_difference"]].rename(columns={"important_difference": "label"}).to_csv(
    basedir / "binary_important_diff_labels.csv", index=False)

# original diff
avail_df[["bag_id", "difference"]].rename(columns={"difference": "label"}).to_csv(
    basedir / "difference_labels.csv", index=False)
